In [ ]:
# Notebook 03 - Feature Engineering (step-by-step)

# Cell A - Setup & Config
# - Import libs: pandas, numpy, ast, sklearn, scipy.sparse, pathlib, json, joblib
# - Define config: input/output paths, seed, TF-IDF params, topk_similarity

# Cell B - Load processed data
# - Load data/processed/recipes_clean.csv
# - Load data/processed/interactions_clean.csv
# - Parse list-like columns: ingredients_clean, tags_clean

# Cell C - Schema checks & hard guards
# - Validate dtypes: id, user_id, recipe_id, rating, minutes, calories
# - Remove orphan interactions (recipe_id not in recipes)
# - Enforce feature ranges (minutes > 0, calories in [10, 5000] or flag outlier)

# Cell D - Anti-leakage split
# - Split interactions: train/validation (time-based or leave-last-1 per user)
# - Use train_interactions to compute user/popularity statistics
# - Do not use validation interactions when fitting user-level stats

# Cell E - Text feature construction
# - Build ingredients_text, tags_text, description_clean
# - Build combined_text with deterministic weighting strategy

# Cell F - TF-IDF recipe vectors
# - Fit TfidfVectorizer on combined_text
# - Build recipe_tfidf_matrix + recipe_id/index mapping

# Cell G - Content similarity artifacts
# - Compute top-k cosine neighbors per recipe (avoid full NxN persistence)
# - Output columns: recipe_id, neighbor_recipe_id, similarity

# Cell H - Numerical & constraint features
# - Build rule-based fields: calories, minutes, n_ingredients_calc
# - Build buckets/flags: quick_meal, low_calorie, etc.

# Cell I - User features (from train only)
# - avg_rating_given, rating_count, activity_level
# - favorite_tags from high-rated history
# - default fallback for new users

# Cell J - Recipe popularity features
# - rating_count, rating_mean, popularity_score (Bayesian smoothing)
# - is_cold_item flag for cold-start logic

# Cell K - Feature validation checklist
# - Validate nulls, schema, key uniqueness
# - Coverage checks: users/features, recipes/vectors, recipes/top-k neighbors
# - Print final sanity summary

# Cell L - Save artifacts + manifest
# - Save into artifacts/features/
#   tfidf_vectorizer.joblib
#   recipe_tfidf_matrix.npz
#   recipe_index_map.parquet
#   recipe_similarity_topk.parquet
#   recipe_features.parquet
#   user_features.parquet
#   popularity_features.parquet
#   feature_manifest.json

# Output
# - Print artifact paths, shapes, coverage, and build timestamp

In [2]:
import pandas as pd
import numpy as np
import ast
import json
import joblib
from pathlib import Path
from datetime import datetime
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
recipes = pd.read_csv("../data/processed/recipes_clean.csv")
interactions = pd.read_csv("../data/processed/interactions_clean.csv")

def parse_list_safe(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x.strip())
            return v if isinstance(v, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

for col in ["ingredients_clean", "tags_clean", "ingredients", "tags", "steps"]:
    if col in recipes.columns:
        recipes[col] = recipes[col].apply(parse_list_safe)

print(f"recipes:      {recipes.shape}")
print(f"interactions: {interactions.shape}")
print(f"Sample ingredients_clean: {recipes['ingredients_clean'].iloc[0][:3]}")
print(f"Sample tags_clean:        {recipes['tags_clean'].iloc[0][:3]}")


recipes:      (77300, 24)
interactions: (706514, 5)
Sample ingredients_clean: ['winter squash', 'mexican seasoning', 'mixed spice']
Sample tags_clean:        ['60-minutes-or-less', 'time-to-make', 'course']


In [ ]:
# === Cell C - Schema checks & hard guards ===

print("--- Before guards ---")
print(f"recipes:      {recipes.shape}")
print(f"interactions: {interactions.shape}")

# 1) Ép kiểu dữ liệu core
recipes["id"] = pd.to_numeric(recipes["id"], errors="coerce").astype("Int64")
recipes["minutes"] = pd.to_numeric(recipes["minutes"], errors="coerce")
recipes["calories"] = pd.to_numeric(recipes["calories"], errors="coerce")

interactions["user_id"] = pd.to_numeric(interactions["user_id"], errors="coerce").astype("Int64")
interactions["recipe_id"] = pd.to_numeric(interactions["recipe_id"], errors="coerce").astype("Int64")
interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce")
if "date" in interactions.columns:
    interactions["date"] = pd.to_datetime(interactions["date"], errors="coerce")

# 2) Loại orphan interactions (recipe_id không có trong recipes)
valid_ids = set(recipes["id"].dropna())
n_before = len(interactions)
interactions = interactions[interactions["recipe_id"].isin(valid_ids)].copy()
print(f"Orphan interactions removed: {n_before - len(interactions):,}")

# 3) Filter range hợp lý cho feature
recipes = recipes[recipes["minutes"].between(1, 1440)].copy()
recipes["is_calorie_outlier"] = ~recipes["calories"].between(10, 5000)
n_outlier = recipes["is_calorie_outlier"].sum()
print(f"Calorie outliers flagged (kept but flagged): {n_outlier:,}")

# 4) Sync lại sau filter
interactions = interactions[interactions["recipe_id"].isin(set(recipes["id"]))].copy()

print("\n--- After guards ---")
print(f"recipes:      {recipes.shape}")
print(f"interactions: {interactions.shape}")
print(f"unique users: {interactions['user_id'].nunique():,}")
print(f"unique items: {interactions['recipe_id'].nunique():,}")
print(f"orphan check: {len(set(interactions['recipe_id']) - set(recipes['id']))} orphans")

In [ ]:
# === Cell D - Anti-leakage split ===
# Split interactions theo thời gian: 80% cũ nhất -> train, 20% mới nhất -> validation
# Chỉ dùng train_interactions để tính user/popularity stats phía sau

interactions_sorted = interactions.sort_values("date").reset_index(drop=True)
split_idx = int(len(interactions_sorted) * 0.8)

train_interactions = interactions_sorted.iloc[:split_idx].copy()
val_interactions = interactions_sorted.iloc[split_idx:].copy()

train_user_ids = set(train_interactions["user_id"])
train_recipe_ids = set(train_interactions["recipe_id"])
val_user_ids = set(val_interactions["user_id"])
val_recipe_ids = set(val_interactions["recipe_id"])

cold_start_users = val_user_ids - train_user_ids
cold_start_items = val_recipe_ids - train_recipe_ids

print(f"train_interactions: {len(train_interactions):,} rows")
print(f"val_interactions:   {len(val_interactions):,} rows")
print(f"train date range:   {train_interactions['date'].min()} -> {train_interactions['date'].max()}")
print(f"val   date range:   {val_interactions['date'].min()} -> {val_interactions['date'].max()}")
print(f"\ntrain users: {len(train_user_ids):,}  |  train items: {len(train_recipe_ids):,}")
print(f"val   users: {len(val_user_ids):,}  |  val   items: {len(val_recipe_ids):,}")
print(f"cold-start users in val: {len(cold_start_users):,}")
print(f"cold-start items in val: {len(cold_start_items):,}")